-----

### El Motor Isométrico - Simulación de Profundidad y Lucha Libre**

**Objetivo:** Extender nuestro motor para soportar una perspectiva isométrica. Se implementarán las funciones de transformación de coordenadas necesarias, se adaptarán los objetos del juego para manejar la profundidad (Z-sorting) y se construirá una escena de ejemplo con personajes que se mueven en un plano isométrico, como un ring de lucha libre.

-----

### **1. Discusión: La Ilusión de la Profundidad (Isometría)**

Hasta ahora, nuestros juegos han sido "planos". En el plataformero, los ejes son `(x, y)` donde `y` es la altura. En una vista isométrica, los ejes del mundo siguen siendo `(x, y)` en una rejilla lógica, pero se **dibujan** en la pantalla de una forma que simula la profundidad.

  * **La Rejilla de Diamante:** El mundo del juego ya no se piensa como una cuadrícula de cuadrados, sino como una cuadrícula de **diamantes** (rombos). Esto crea la sensación de estar viendo el mundo desde un ángulo.
  * **El Eje "Falso Z":** El movimiento "hacia arriba" en la pantalla ahora corresponde a moverse "hacia el fondo" en el mundo del juego. La coordenada `y` de la pantalla se convierte en nuestro indicador de profundidad o "eje Z".
  * **El Desafío del Z-Sorting (Orden de Profundidad):** Este es el concepto más importante. Para que la ilusión funcione, los objetos que están "más al sur" (con una coordenada `y` mayor) deben dibujarse **después** (y por lo tanto, **encima**) de los que están "más al norte". Si no lo hacemos, un personaje que está detrás de otro se dibujará por encima, rompiendo la perspectiva.

**Ejemplos en juegos conocidos:**

  * **Diablo II / Path of Exile:** Clásicos del Action RPG que usan la isometría para crear vastos mundos y dar una sensación de espacio táctico.
  * **Final Fantasy Tactics / X-COM:** Juegos de estrategia donde la rejilla isométrica es fundamental para el posicionamiento, la cobertura y la línea de visión.
  * **Stardew Valley / Animal Crossing:** Usan una perspectiva similar (aunque no es isometría pura) para dar a sus mundos una sensación de profundidad y encanto.

-----

### **2. Expansión del Motor: El Módulo de Isometría**

Añadiremos un nuevo módulo a nuestra carpeta `engine/` que contendrá todas las matemáticas necesarias.

#### **`engine/iso_tools.py` (¡NUEVO\!)**

Este módulo contendrá las funciones para convertir coordenadas del "mundo lógico" (una simple rejilla) a las coordenadas de "pantalla" (isométricas).

```python
# engine/iso_tools.py


def world_to_iso(x, y, tile_width, tile_height):
    """Convierte coordenadas de una rejilla de mundo a coordenadas isométricas de pantalla."""
    iso_x = (x - y) * (tile_width / 2)
    iso_y = (x + y) * (tile_height / 2)
    return iso_x, iso_y

def iso_to_world(iso_x, iso_y, tile_width, tile_height):
    """Convierte coordenadas de pantalla isométricas a coordenadas de la rejilla del mundo."""
    # Esta es la matemática inversa, útil para la detección del ratón
    world_x = (iso_x / (tile_width / 2) + iso_y / (tile_height / 2)) / 2
    world_y = (iso_y / (tile_height / 2) - iso_x / (tile_width / 2)) / 2
    return world_x, world_y
```

-----

### **3. Creando el Juego de Lucha Libre**

Ahora, crearemos un nuevo proyecto que utilice nuestro motor y las nuevas herramientas isométricas.

#### **`wrestling/entities/wrestler.py`**

Nuestro luchador será un `GameObject`, pero su lógica de movimiento y dibujado será especial.

```python
# wrestling/entities/wrestler.py
import pygame
from engine.game_object import GameObject
from engine.iso_tools import world_to_iso

class Wrestler(GameObject):
    def __init__(self, grid_x, grid_y, frames, tile_width, tile_height):
        # La posición inicial es en coordenadas de la rejilla del mundo
        self.grid_pos = pygame.math.Vector2(grid_x, grid_y)
        self.tile_width = tile_width
        self.tile_height = tile_height
        
        # Calculamos la posición inicial en la pantalla
        screen_x, screen_y = world_to_iso(self.grid_pos.x, self.grid_pos.y, tile_width, tile_height)
        
        super().__init__(screen_x, screen_y, frames)
        self.speed = 0.05 # Movimiento en la rejilla

    def update(self, *args, **kwargs):
        super().update() # Maneja la animación

        keys = pygame.key.get_pressed()
        move_vec = pygame.math.Vector2(0, 0)

        # El movimiento de las flechas ahora afecta a la rejilla del mundo
        if keys[pygame.K_UP]:
            move_vec.y -= 1 # Moverse "hacia atrás" en el mundo
            move_vec.x -= 1
        if keys[pygame.K_DOWN]:
            move_vec.y += 1 # Moverse "hacia adelante" en el mundo
            move_vec.x += 1
        if keys[pygame.K_LEFT]:
            move_vec.y += 1
            move_vec.x -= 1
        if keys[pygame.K_RIGHT]:
            move_vec.y -= 1
            move_vec.x += 1
            
        if move_vec.length() > 0:
            self.grid_pos += move_vec.normalize() * self.speed

        # Actualizamos la posición en pantalla basándonos en la nueva posición del mundo
        screen_x, screen_y = world_to_iso(self.grid_pos.x, self.grid_pos.y, self.tile_width, self.tile_height)
        self.rect.center = (screen_x, screen_y)
```

#### **`wrestling/scenes/ring_scene.py`**

Esta escena crea el ring, los luchadores y, lo más importante, maneja el **Z-sorting** en el bucle de dibujado.

```python
# wrestling/scenes/ring_scene.py
import pygame
from engine.ui import Scene
from ..entities.wrestler import Wrestler

class RingScene(Scene):
    def __init__(self, game):
        super().__init__(game)

        # Cargamos los assets
        # Asumimos que tienes un 'luchador.json' con una animación 'idle'
        luchador_frames = self.game.assets.get_animation('luchador_ss', 'idle')
        
        self.TILE_WIDTH = 64
        self.TILE_HEIGHT = 32

        # Creamos los luchadores en posiciones de la rejilla del mundo
        self.luchador1 = Wrestler(5, 5, luchador_frames, self.TILE_WIDTH, self.TILE_HEIGHT)
        self.luchador2 = Wrestler(7, 7, luchador_frames, self.TILE_WIDTH, self.TILE_HEIGHT)
        
        # Un grupo para poder actualizarlos a todos a la vez
        self.all_sprites = pygame.sprite.Group(self.luchador1, self.luchador2)
        
        # Creamos la imagen del ring (un rombo)
        self.ring_image = self.create_ring_surface(10, 10)
        self.ring_rect = self.ring_image.get_rect(center=self.game.screen.get_rect().center)

    def create_ring_surface(self, grid_width, grid_height):
        # Crea una superficie para dibujar el ring isométrico
        width = (grid_width + grid_height) * (self.TILE_WIDTH / 2)
        height = (grid_width + grid_height) * (self.TILE_HEIGHT / 2)
        surface = pygame.Surface((width, height), pygame.SRCALPHA)
        
        for y in range(grid_height):
            for x in range(grid_width):
                iso_x, iso_y = world_to_iso(x, y, self.TILE_WIDTH, self.TILE_HEIGHT)
                # Centramos el dibujo en la superficie
                iso_x += (grid_width * self.TILE_WIDTH / 2) - (self.TILE_WIDTH / 2)
                
                # Dibujamos un tile de rombo
                points = [
                    (iso_x, iso_y + self.TILE_HEIGHT / 2),
                    (iso_x + self.TILE_WIDTH / 2, iso_y),
                    (iso_x + self.TILE_WIDTH, iso_y + self.TILE_HEIGHT / 2),
                    (iso_x + self.TILE_WIDTH / 2, iso_y + self.TILE_HEIGHT),
                ]
                pygame.draw.polygon(surface, (50, 50, 80), points)
                pygame.draw.polygon(surface, (200, 200, 200), points, 1)

        return surface

    def update(self):
        self.all_sprites.update()

    def draw(self, surface):
        surface.fill((30, 30, 30))
        surface.blit(self.ring_image, self.ring_rect)

        # --- CAMBIO CLAVE: Z-SORTING ---
        # Antes de dibujar, ordenamos los sprites basándonos en su coordenada Y.
        # Los que tienen una Y mayor (están más abajo) se dibujan al final (encima).
        for sprite in sorted(self.all_sprites, key=lambda s: s.rect.centery):
            # Centramos todo el mundo en la pantalla
            offset_x = self.game.screen.get_width() / 2 - self.ring_rect.width / 2
            offset_y = 100 # Un pequeño offset vertical
            
            # Dibujamos el sprite en su posición de pantalla calculada + el offset general
            surface.blit(sprite.image, (sprite.rect.x + offset_x, sprite.rect.y + offset_y))
            
    def handle_events(self, events):
        pass
```

*(Los archivos `game.py` y `main.py` seguirían la misma estructura que en el plataformero, pero cargarían la `RingScene` y los assets del luchador).*

### **Resumen de la Implementación**

1.  **Motor Isométrico (`iso_tools.py`):** Un módulo simple pero potente con las fórmulas matemáticas para convertir entre coordenadas de mundo y de pantalla.
2.  **Entidad Isométrica (`Wrestler`):** Mantiene su posición lógica en una rejilla (`grid_pos`) y, en cada `update`, calcula su posición de dibujado en la pantalla usando las herramientas isométricas. El input del jugador modifica la posición en la rejilla, no directamente en la pantalla.
3.  **Z-Sorting en el Dibujado:** En `RingScene.draw()`, la línea `sorted(self.all_sprites, key=lambda s: s.rect.centery)` ordena la lista de sprites antes de dibujarlos, garantizando que la perspectiva se mantenga correctamente.



**Parte 2 (Re load)**

-----

### **El Motor Isométrico y el Juego de Lucha**

**Objetivo:** Extender nuestro motor para que soporte una perspectiva isométrica, permitiendo la creación de juegos con sensación de profundidad. Explicaremos las diferencias conceptuales, actualizaremos el motor y construiremos un prototipo de juego de lucha libre.

-----

### **1. Discusión Teórica: Perspectiva Lateral vs. Isométrica**

Es fundamental entender estas dos formas de "ver" un mundo 2D.

#### **Vista Lateral (Side-Scroller)**

  * **Qué es:** Es la vista clásica de juegos como *Super Mario Bros.*, donde vemos el mundo desde un lado, como si fuera un escenario de teatro.
  * **Coordenadas:** El eje **X** controla el movimiento horizontal (izquierda/derecha). El eje **Y** controla el movimiento vertical (arriba/abajo).
  * **Profundidad:** No existe. Los objetos simplemente se dibujan unos encima de otros.
  * **Gravedad:** Actúa directamente sobre el eje **Y** (un valor positivo constante que se suma a la velocidad vertical).
  * **Colisiones:** Simples y directas (rectángulos que chocan).

#### **Perspectiva Isométrica**

  * **Qué es:** Es una técnica 2D para simular una vista 3D. Vemos el mundo desde una esquina, "en ángulo", lo que nos permite ver el suelo y la profundidad.
  * **Coordenadas:** Aquí está el truco. El juego **lógicamente** sigue ocurriendo en una rejilla 2D simple (como un tablero de ajedrez), que llamaremos `(grid_x, grid_y)`. Pero estas coordenadas se **traducen** a coordenadas de pantalla (`screen_x`, `screen_y`) mediante una fórmula matemática.
  * **Profundidad (Z-Sorting):** Este es el concepto más importante. En la pantalla, el eje **Y** (arriba/abajo) ahora representa la **profundidad**. Un personaje con una coordenada `screen_y` más baja está "más cerca" de la cámara y debe dibujarse **encima** de un personaje con una `screen_y` más alta.
  * **Gravedad:** Generalmente no se usa de la misma manera, ya que los personajes se mueven sobre un plano de suelo.

-----

### **2. `GameLoop` vs. `SceneManager`: Dónde Ocurre la Lógica**


  * **GameLoop (`game.py`):** Es el **corazón** del juego. Es el `while self.running:` que se ejecuta 60 veces por segundo. Su única responsabilidad es "bombear":

    1.  Obtener todos los eventos (`pygame.event.get()`).
    2.  Llamar al método `update()` de la escena activa.
    3.  Llamar al método `draw()` de la escena activa.
        No sabe *qué* está pasando en el juego, solo que *algo* debe pasar.

  * **SceneManager (Patrón en `game.py`):** Es el **cerebro**. La clase de tu juego (`WrestlingGame`) actúa como el gestor de escenas. Mantiene un diccionario de escenas (`self.scenes`) y una variable (`self.current_scene`) que le dice al `GameLoop` a quién darle el control.

  * **Lógica de Colisiones y Físicas (`entities/player.py`):** La lógica de colisiones **se ejecuta** durante la fase de `update` del `GameLoop`. Específicamente:

    1.  `GameLoop.run()` llama a `self.current_scene.update()`.
    2.  `GameScene.update()` llama a `self.all_sprites.update(...)`.
    3.  `pygame` llama al método `update()` de cada sprite, incluido tu `Player`.
    4.  **Dentro de `Player.update()`** es donde se comprueban las teclas y se ejecutan las físicas y las colisiones (`pygame.sprite.spritecollide`).

La lógica no "flota", está anidada dentro de la llamada de `update()` de la escena activa.

-----

### **3. Preparando los Assets: Tiled y Sprites Isométricos**

Para este proyecto, necesitas assets específicos. Los que subiste no son isométricos. Te recomiendo buscar assets de **Kenney**, que son gratuitos y de alta calidad.

  * **Tileset del Ring:** Busca en **[Kenney.nl](https://kenney.nl/assets/tower-defense-kit)**. El "Tower Defense Kit" contiene tiles isométricos de suelo perfectos.
  * **Luchadores:** Busca en **[OpenGameArt.org](https://opengameart.org/)** "isometric character".

#### **Paso a Paso en Tiled (con Traducción)**

1.  **Nuevo Mapa:** `Archivo > Nuevo > Nuevo Mapa...`.
2.  **Configuración (¡Importante\!):**
      * **Orientación:** Selecciona **`Isométrico`**.
      * **Tamaño del tile:** Los tiles isométricos no son cuadrados. Definen el ancho y alto del "diamante". Un estándar común es **64x32** (ancho x alto).
      * **Tamaño del mapa:** ej. 30x30 tiles.
3.  **Añadir Tileset:** En el panel "Patrones" (abajo a la derecha), haz clic en "Nuevo Tileset" y selecciona tu imagen de tileset. Asegúrate de que Tiled sepa que el tamaño del tile es 64x32.
4.  **Propiedades de Colisión:**
      * Haz clic en el botón "Editar Tileset" (la llave inglesa).
      * Selecciona el tile de la cuerda del ring.
      * En el panel "Propiedades" (izquierda), añade una propiedad `bool` llamada **`collides`** y **marca la casilla**.
5.  **Crear Capas:**
      * `Capa de Patrones` (Tile Layer): Nómbrala **`Suelo_colision`**. (El sufijo `_colision` es detectado por nuestro motor).
      * `Capa de Patrones` (Tile Layer): Nómbrala **`Cuerdas_colision`**.
      * `Capa de Objetos` (Object Layer): Nómbrala **`Entidades`**.
6.  **Pintar:** Pinta el suelo en la capa `Suelo_colision` y las cuerdas en `Cuerdas_colision`.
7.  **Posicionar al Jugador:** Selecciona la capa `Entidades`, usa la herramienta **"Insertar Objeto"** (el rectángulo punteado) y coloca un objeto. Dale el **Nombre:** `player_start`.
8.  **Exportar (¡Crucial\!):**
      * `Archivo > Propiedades del mapa...` -\> Marca la casilla **"Incrustar conjunto de patrones en el mapa"**.
      * `Archivo > Exportar como...` -\> Guarda como tipo **JSON** (ej: `ring.json`).

-----

### **4. Expansión del Motor (Nuevos Módulos)**

Añadimos un nuevo módulo para las matemáticas isométricas.

#### **Archivo: `engine/iso_tools.py` (¡NUEVO\!)**

```python
# engine/iso_tools.py
import pygame

class IsoTools:
    """Utilidad para convertir entre coordenadas de mundo y pantalla isométrica."""
    def __init__(self, tile_width, tile_height):
        self.tile_width = tile_width
        self.tile_height = tile_height
        self.half_width = tile_width / 2
        self.half_height = tile_height / 2

    def world_to_iso(self, grid_x, grid_y):
        """Convierte coordenadas de la rejilla (lógica) a la pantalla (dibujado)."""
        iso_x = (grid_x - grid_y) * self.half_width
        iso_y = (grid_x + grid_y) * self.half_height
        return iso_x, iso_y

    def iso_to_world(self, screen_x, screen_y):
        """Convierte coordenadas de la pantalla (clic del ratón) a la rejilla (lógica)."""
        world_x = (screen_x / self.half_width + screen_y / self.half_height) / 2
        world_y = (screen_y / self.half_height - screen_x / self.half_width) / 2
        return world_x, world_y
```

-----

### **5. El Código del Juego de Lucha Libre (`wrestling/`)**

Aquí creamos el nuevo juego `wrestling` que usa el motor y las nuevas herramientas.

#### **Archivo: `wrestling/entities/wrestler.py`**

```python
import pygame
from engine.game_object import GameObject
from engine.iso_tools import IsoTools

class Wrestler(GameObject):
    """
    Un personaje que se mueve en un plano isométrico.
    Mantiene una posición lógica (grid_pos) y una posición visual (pos).
    """
    def __init__(self, grid_x, grid_y, assets, iso_tools):
        self.assets = assets
        self.animations = {
            'idle': self.assets.get_animation('luchador_ss', 'idle')
            # Aquí cargarías 'walk_n', 'walk_s', 'walk_e', 'walk_w'
        }
        
        self.iso_tools = iso_tools
        self.grid_pos = pygame.math.Vector2(grid_x, grid_y)
        
        # Calculamos la posición inicial en pantalla
        screen_x, screen_y = self.iso_tools.world_to_iso(grid_x, grid_y)
        
        # Inicializamos el GameObject con la primera imagen
        super().__init__(screen_x, screen_y, self.animations['idle'])
        
        # Ajustamos el rect para que los "pies" estén en la coordenada
        self.rect.midbottom = (screen_x, screen_y)
        self.pos = pygame.math.Vector2(self.rect.center)
        self.speed = 2 # Velocidad de movimiento en la rejilla (tiles por segundo)

    def update(self, dt, plataformas, *args, **kwargs):
        move_vec = pygame.math.Vector2(0, 0)
        teclas = pygame.key.get_pressed()
        
        if teclas[pygame.K_UP]:    move_vec.y -= 1
        if teclas[pygame.K_DOWN]:  move_vec.y += 1
        if teclas[pygame.K_LEFT]:  move_vec.x -= 1
        if teclas[pygame.K_RIGHT]: move_vec.x += 1
            
        if move_vec.length() > 0:
            self.grid_pos += move_vec.normalize() * dt * self.speed

        # Actualizamos la posición objetivo en la pantalla
        target_screen_x, target_screen_y = self.iso_tools.world_to_iso(self.grid_pos.x, self.grid_pos.y)
        target_pos = pygame.math.Vector2(target_screen_x, target_screen_y)
        
        # Usamos LERP para que el sprite se mueva suavemente a su posición
        self.pos = self.pos.lerp(target_pos, 0.1)

        # Actualizamos la animación y el rect final
        super().update() # Llama a update_animation
        
        # Redimensionamos la imagen y la centramos por los pies
        self.image = pygame.transform.scale(self.image, (64, 96)) # Asumimos un tamaño
        self.rect = self.image.get_rect(midbottom=self.pos)
        self.depth = self.rect.bottom # ¡Muy importante para el Z-Sorting!
```

#### **Archivo: `wrestling/scenes/game_scene.py`**

```python
import pygame
import os
from engine.ui import Scene
from engine.tilemap import TilemapLoader
from engine.iso_tools import IsoTools
from wrestling.entities.wrestler import Wrestler

class GameScene(Scene):
    def __init__(self, game):
        super().__init__(game)
        
        # 1. Cargamos el mapa de Tiled
        loader = TilemapLoader(self.game.assets)
        level_path = os.path.join(self.game.assets_dir, 'ring.json')
        result = loader.load_level(level_path)
        
        if result is None: self.game.running = False; return
        self.sprite_groups, world_w, world_h, player_start_pos = result

        # 2. Creamos las herramientas isométricas
        # Usamos el tamaño de tile definido en Tiled
        self.iso_tools = IsoTools(tile_width=64, tile_height=32)

        # 3. Creamos el jugador
        # Convertimos la posición de Tiled (píxeles) a la rejilla lógica
        start_grid_x, start_grid_y = self.iso_tools.iso_to_world(player_start_pos[0], player_start_pos[1])
        self.jugador = Wrestler(start_grid_x, start_grid_y, self.game.assets, self.iso_tools)
        self.sprite_groups["all_sprites"].add(self.jugador)

        # 4. Centramos la "cámara" (es solo un offset de dibujado)
        self.camera_offset = [self.game.screen.get_width() // 2, 200]

    def update(self):
        dt = self.game.clock.get_time() / 1000.0 # Delta time en segundos
        # Pasamos dt y los sprites de colisión al update
        self.sprite_groups["all_sprites"].update(dt=dt, plataformas=self.sprite_groups["collision_sprites"])

    def draw(self, surface):
        surface.fill((30, 10, 10)) # Fondo oscuro

        # --- Z-SORTING (Orden de Profundidad) ---
        # Ordenamos todos los sprites basándonos en su 'self.depth'
        # El que tenga 'depth' más grande (más abajo en pantalla) se dibuja último (más al frente).
        for sprite in sorted(self.sprite_groups["all_sprites"], key=lambda s: s.depth):
            # Aplicamos el offset de la cámara para centrar el mundo
            screen_pos = sprite.rect.move(self.camera_offset[0], self.camera_offset[1])
            surface.blit(sprite.image, screen_pos)
            
    def handle_events(self, events):
        pass
```

#### **Archivo: `wrestling/game.py`**

```python
import pygame
import os
from engine.asset_manager import AssetManager
from engine.dialogue import DialogueManager
from wrestling.scenes.game_scene import GameScene

class WrestlingGame:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((1280, 720))
        pygame.display.set_caption("Lucha Libre Isométrica")
        self.clock = pygame.time.Clock()
        self.running = True
        self.game_dir = os.path.dirname(__file__)
        self.assets_dir = os.path.join(self.game_dir, 'assets', 'sprites')
        self.assets = AssetManager()
        self.dialogue_manager = DialogueManager(self.screen) # No lo usamos aquí, pero es parte del motor
        self.load_assets()
        
        if self.running:
            self.scenes = {'game': GameScene(self)}
            self.current_scene = self.scenes['game']

    def load_assets(self):
        try:
            # Asegúrate de tener un 'luchador.json' y un 'ring.json'
            self.assets.load_spritesheet('luchador_ss', os.path.join(self.assets_dir, 'luchador.json'))
        except Exception as e:
            print(f"Error fatal al cargar assets: {e}"); self.running = False
            
    def run(self):
        if not self.running: return
        while self.running:
            events = pygame.event.get()
            for event in events:
                if event.type == pygame.QUIT: self.running = False
            self.current_scene.handle_events(events)
            self.current_scene.update()
            self.current_scene.draw(self.screen)
            pygame.display.flip()
        pygame.quit()

    def change_scene(self, scene_name):
        self.current_scene = self.scenes.get(scene_name)
```

#### **Archivo: `wrestling/main.py`**

```python
# wrestling/main.py
from wrestling.game import WrestlingGame

def main():
    juego = WrestlingGame()
    juego.run()

if __name__ == '__main__':
    main()
```